# Лабораторная работа №2

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum, avg, count, row_number, concat, lit, year, month, 
    dense_rank, when
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Lab2_ETL_Reports") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.7.2,"
            "com.clickhouse:clickhouse-jdbc:0.4.6") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("SparkSession created")

SparkSession created


In [2]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 3.0 MB/s eta 0:00:00a 0:00:010m


In [3]:
pg_url = "jdbc:postgresql://postgres:5432/lab2_db"
pg_props = {
    "user": "pg",
    "password": "pg",
    "driver": "org.postgresql.Driver",
}

ch_url = "jdbc:clickhouse://clickhouse:8123/default"
ch_props = {
    "user": "default",
    "password": "",
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "batchsize": "1000",
    "isolationLevel": "NONE"
}

print("Configurated")

Configurated


In [4]:
df_raw = spark.read.jdbc(url=pg_url, table="mock_data", properties=pg_props)
df_raw = df_raw.dropDuplicates()
print(f"Load data with rows num: {df_raw.count()}")
df_raw.show(3)

Load data with rows num: 10000
+---+-------------------+------------------+------------+--------------------+----------------+--------------------+-----------------+-----------------+------------------+-----------------+----------------+--------------------+--------------+------------------+------------+----------------+-------------+----------------+----------+----------------+--------------+---------------+-------------+----------------+----------+--------------+------------+-----------+-------------+------------+--------------------+------------+--------------+-------------+------------+-------------+----------------+--------------------+--------------+---------------+--------------------+-------------------+-------------+-----------------+--------------------+--------------+----------------+-----------------+----------------+
| id|customer_first_name|customer_last_name|customer_age|      customer_email|customer_country|customer_postal_code|customer_pet_type|customer_pet_name|custom

In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, sum, avg, count, row_number, lit, year, month,
    dense_rank, when
)

w_name    = Window.orderBy("name")
w_product = Window.orderBy("product_id")

dim_brands = df_raw.select(col("product_brand").alias("name")) \
    .dropDuplicates(["name"]) \
    .withColumn("brand_id", row_number().over(w_name))

dim_suppliers = df_raw.select(
    col("supplier_name").alias("name"),
    col("supplier_contact").alias("contact"),
    col("supplier_email").alias("email"),
    col("supplier_phone").alias("phone"),
    col("supplier_address").alias("address"),
    col("supplier_city").alias("city"),
    col("supplier_country").alias("country")
).dropDuplicates(["name"]).withColumn("supplier_id", row_number().over(w_name))

dim_stores = df_raw.select(
    col("store_name").alias("name"),
    col("store_location").alias("location"),
    col("store_city").alias("city"),
    col("store_state").alias("state"),
    col("store_country").alias("country"),
    col("store_phone").alias("phone"),
    col("store_email").alias("email")
).dropDuplicates(["name"]).withColumn("store_id", row_number().over(w_name))

dim_customers = df_raw.select(
    col("sale_customer_id").alias("customer_id"),
    col("customer_first_name").alias("first_name"),
    col("customer_last_name").alias("last_name"),
    col("customer_age").alias("age"),
    col("customer_email").alias("email"),
    col("customer_country").alias("country"),
    col("customer_postal_code").alias("postal_code"),
    col("customer_pet_type").alias("pet_type"),
    col("customer_pet_name").alias("pet_name"),
    col("customer_pet_breed").alias("pet_breed")
).dropDuplicates(["customer_id"])

dim_sellers = df_raw.select(
    col("sale_seller_id").alias("seller_id"),
    col("seller_first_name").alias("first_name"),
    col("seller_last_name").alias("last_name"),
    col("seller_email").alias("email"),
    col("seller_country").alias("country"),
    col("seller_postal_code").alias("postal_code")
).dropDuplicates(["seller_id"])


dim_products_raw = df_raw.select(
    col("sale_product_id").alias("product_id"),
    col("product_name").alias("name"),
    col("product_category").alias("category"),
    col("product_price").alias("price"),
    col("product_weight").alias("weight"),
    col("product_color").alias("color"),
    col("product_size").alias("size"),
    col("product_material").alias("material"),
    col("product_description").alias("description"),
    col("product_rating").alias("rating"),
    col("product_brand"),
    col("supplier_name")
).dropDuplicates(["product_id"])

print("All dimensions are generated")

All dimensions are generated


In [6]:
import psycopg2

def pg_execute(sql):
    with psycopg2.connect(host="postgres", port=5432, database="lab2_db", user="pg", password="pg") as conn:
        conn.autocommit = True
        with conn.cursor() as cur:
            cur.execute(sql)

print("Clearing...")
for tbl in ["fact_sales", "dim_products", "dim_sellers", "dim_customers", "dim_stores", "dim_suppliers", "dim_brands"]:
    pg_execute(f"TRUNCATE TABLE {tbl} RESTART IDENTITY CASCADE")
print("Cleared")

print("Load measurements: ")
dim_brands.write.jdbc(url=pg_url, table="dim_brands", mode="append", properties=pg_props)
print("dim_brands done")
dim_suppliers.write.jdbc(url=pg_url, table="dim_suppliers", mode="append", properties=pg_props)
print("dim_suppliers done")
dim_stores.write.jdbc(url=pg_url, table="dim_stores", mode="append", properties=pg_props)
print("dim_stores done")
dim_customers.write.jdbc(url=pg_url, table="dim_customers", mode="append", properties=pg_props)
print("dim_customers done")
dim_sellers.write.jdbc(url=pg_url, table="dim_sellers", mode="append", properties=pg_props)
print("dim_sellers done")


d_brands = spark.read.jdbc(url=pg_url, table="dim_brands", properties=pg_props)
d_suppliers = spark.read.jdbc(url=pg_url, table="dim_suppliers", properties=pg_props)

dim_products = dim_products_raw \
    .join(d_brands.select(col("name").alias("brand_name"), col("brand_id")),
          dim_products_raw.product_brand == col("brand_name"), "left") \
    .join(d_suppliers.select(col("name").alias("sup_name"), col("supplier_id")),
          dim_products_raw.supplier_name == col("sup_name"), "left") \
    .select("product_id", "name", "category", "price", "weight", "color",
            "size", "material", "description", "rating", "supplier_id", "brand_id")

dim_products.write.jdbc(url=pg_url, table="dim_products", mode="append", properties=pg_props)
print("dim_products done")


d_stores = spark.read.jdbc(url=pg_url, table="dim_stores", properties=pg_props)
d_customers = spark.read.jdbc(url=pg_url, table="dim_customers", properties=pg_props)
d_sellers = spark.read.jdbc(url=pg_url, table="dim_sellers", properties=pg_props)
d_products = spark.read.jdbc(url=pg_url, table="dim_products", properties=pg_props)

fact_sales = df_raw \
    .join(d_products.select("product_id"), df_raw.sale_product_id == col("product_id"), "left") \
    .join(d_customers.select("customer_id"), df_raw.sale_customer_id == col("customer_id"), "left") \
    .join(d_sellers.select("seller_id"), df_raw.sale_seller_id == col("seller_id"), "left") \
    .join(d_stores.select(col("store_id"), col("name").alias("st_name")),
          df_raw.store_name == col("st_name"), "left") \
    .select(
        col("sale_customer_id").alias("sale_id"),
        col("sale_date"), col("customer_id"), col("seller_id"),
        col("product_id"), col("store_id"),
        col("sale_quantity").alias("quantity"),
        col("sale_total_price").alias("total_price")
    ).dropDuplicates(["sale_id"])

fact_sales.write.jdbc(url=pg_url, table="fact_sales", mode="append", properties=pg_props)
print(f"fact_sales done ({fact_sales.count()} rows)")

Clearing...
Cleared
Load measurements: 
dim_brands done
dim_suppliers done
dim_stores done
dim_customers done
dim_sellers done
dim_products done
fact_sales done (1000 rows)


In [7]:
print("Generating data marts...")

f = spark.read.jdbc(url=pg_url, table="fact_sales", properties=pg_props)
p = spark.read.jdbc(url=pg_url, table="dim_products", properties=pg_props)
c = spark.read.jdbc(url=pg_url, table="dim_customers", properties=pg_props)
s = spark.read.jdbc(url=pg_url, table="dim_stores", properties=pg_props)
sup = spark.read.jdbc(url=pg_url, table="dim_suppliers", properties=pg_props)

# Product sales data mart
report_products = f.join(p, "product_id") \
    .groupBy(p.product_id, p.name, p.category, p.rating) \
    .agg(sum("quantity").alias("total_sales_qty"), sum("total_price").alias("total_revenue"),
         avg("rating").alias("avg_rating"), count("sale_id").alias("reviews_count")) \
    .withColumn("rank", dense_rank().over(Window.orderBy(col("total_sales_qty").desc())))

# Customer sales data mart
report_customers = f.join(c, "customer_id") \
    .groupBy(c.customer_id, c.first_name, c.last_name, c.country) \
    .agg(sum("total_price").alias("total_spent"), count("sale_id").alias("total_orders"),
         avg("total_price").alias("avg_check")) \
    .withColumn("rank", dense_rank().over(Window.orderBy(col("total_spent").desc()))) \
    .select("customer_id", concat(col("first_name"), lit(" "), col("last_name")).alias("full_name"),
            "country", "total_spent", "total_orders", "avg_check", "rank")

# Sales data mart with time
report_time = f.withColumn("sale_year", year("sale_date")).withColumn("sale_month", month("sale_date")) \
    .groupBy("sale_year", "sale_month") \
    .agg(sum("total_price").alias("monthly_revenue"), count("sale_id").alias("monthly_orders"),
         avg("total_price").alias("avg_order_size")) \
    .orderBy("sale_year", "sale_month")

# Store sales data mart
report_stores = f.join(s, "store_id") \
    .groupBy(s.store_id, s.name, s.city, s.country) \
    .agg(sum("total_price").alias("total_revenue"), count("sale_id").alias("total_orders"),
         avg("total_price").alias("avg_check")) \
    .withColumn("rank", dense_rank().over(Window.orderBy(col("total_revenue").desc())))

# Supplier sales data mart
report_suppliers = f.join(p, "product_id").join(sup, p.supplier_id == sup.supplier_id) \
    .groupBy(sup.supplier_id, sup.name, sup.country) \
    .agg(sum("total_price").alias("total_revenue"), count("sale_id").alias("total_orders"),
         avg(p.price).alias("avg_product_price")) \
    .withColumn("rank", dense_rank().over(Window.orderBy(col("total_revenue").desc())))

# Product quality data mart
report_quality = f.join(p, "product_id") \
    .groupBy(p.product_id, p.name, p.rating) \
    .agg(count("sale_id").alias("total_reviews"), sum("quantity").alias("total_sales_qty")) \
    .withColumn("rating_segment", when(col("rating") >= 4.5, "High")
                .when(col("rating") >= 3.5, "Medium")
                .otherwise("Low"))

print("All 6 data marts generated")

Generating data marts...
All 6 data marts generated


In [8]:
ch_url = "jdbc:clickhouse://clickhouse:8123/default"

print("Uploading data marts to ClickHouse")

reports_to_ch = {
    "report_sales_products": report_products,
    "report_sales_customers": report_customers,
    "report_sales_time": report_time,
    "report_sales_stores": report_stores,
    "report_sales_suppliers": report_suppliers,
    "report_sales_quality": report_quality,
}


ch_options = {
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "user": "default",
    "password": "clickhouse",
    "compress": "false",
    "decompress": "false",
    "http_connection_provider": "HTTP_URL_CONNECTION",
    "socket_timeout": "60000"
}

for tbl_name, df_report in reports_to_ch.items():
    try:
        df_report.write \
            .format("jdbc") \
            .mode("overwrite") \
            .option("url", ch_url) \
            .option("dbtable", tbl_name) \
            .options(**ch_options) \
            .save()
        print(f"Successfully written: {tbl_name} ({df_report.count()} rows)")
    except Exception as e:
        print(f"Failed to write {tbl_name}: {e}")
        raise

Uploading data marts to ClickHouse
Successfully written: report_sales_products (1000 rows)
Successfully written: report_sales_customers (1000 rows)
Successfully written: report_sales_time (12 rows)
Successfully written: report_sales_stores (352 rows)
Successfully written: report_sales_suppliers (356 rows)
Successfully written: report_sales_quality (1000 rows)
